In [ ]:
# Scraping from Mediatheque Archipel Fouesnant
# Last SCRAPING RUN : Sept 2025 (check last modifications dates on git)

In [ ]:
# Note : you need to have scraped events from the website first.
# You can use "Table Capture" extension for Browser to export the table to CSV.

In [ ]:
# Import libs
import sys
import os
import git

# Ajoute le dossier "ressources" au sys.path
git_root = git.Repo(search_parent_directories=True).working_tree_dir
sys.path.insert(0,   os.path.abspath(  os.path.join(  git_root,'api' ) ) )

import script.libs.utils as utils
import script.libs.scraping_utils as scraping_utils
import script.libs.HttpRequests as HttpRequests
from script.configuration import config, oa
from script.libs.getOaLocation import get_or_create_oa_location

from slugify import slugify
import json
from pprint import pprint

from slugify import slugify
import requests
# import requests_cache
from bs4 import BeautifulSoup
import  dateparser, pytz
from urllib.parse import urlparse
from pprint import pprint
import logging

In [ ]:
# Constant
access_token = oa.getToken()

In [ ]:
# Read existing CSV
file_name="./2025_archipel_fouesnant.csv"
all_events = utils.read_csv(file_name)

In [ ]:
# Constants for Scraping

# locale.setlocale(locale.LC_ALL, 'fr_FR.UTF-8')
DEFAULT_TIME_START = "07h30"
getTimeout= scraping_utils.getTimeout

In [ ]:


def scrap_description(parsed_html:str)->str:
    """Scrap the event page to get the description"""
    if parsed_html.find("div", attrs={"data-view": "event"}):
        description = parsed_html.find("div", attrs={"data-view": "event"}).get_text(separator="\n").strip()
        # logging.info("Using 'data-view=event' for description")
    elif parsed_html.find("div", class_="event-description"):
        description = parsed_html.find("div", id="presentation").get_text(separator="\n").strip()
        logging.info("Using 'id=description' for description")
    else: return None
    # Clean description
    description= description.replace('\n\n', '')
    return description


In [ ]:
def scrap_start_hour(parsed_html:BeautifulSoup)->str:
    """Scrap the event page to get the start hour"""
    if parsed_html.find("div", {"class": "dates"}):
        # logging.info("Using 'div.dates' for start hour")
        hour = parsed_html.find("div", {"class": "dates"}).find_next_sibling("p")
    # elif parsed_html.find("div", class_="event-hour"):
    #     hour_text = parsed_html.find("div", class_="event-hour").get_text()
    else:
        logging.error("No specific event-hour div found.")
        return None
    hour_text=hour.get_text().strip()
    # Extract hour using dateparser
    if hour_text and "H" in hour_text:
        logging.info(f"Parsed start hour from HTML: {hour_text}")
        hour_text= hour_text.replace("H", "h").strip()
        return hour_text
    else:
        logging.warning(f"No valid start hour found, using default: {DEFAULT_TIME_START}")
        return DEFAULT_TIME_START

In [ ]:

def create_event(event:dict)->dict:
        """Get data from existing csv. Returns Dict with OA keys"""
        print("--------------Processing event:", event.get('Titre'),"-----------")
        keywords = event.get('Keyword').split(",")
        event_url=event.get('URL')
        # Default locationId : 11634941
        # Archipel Fouesnant Location ID: 16584185
        # location_uid = get_or_create_oa_location(   event.get('location'),
        #                                                 access_token,
        #                                                 oa.public_key,
        #                                                 f"{config.OA_API_URL}/locations")
        location_uid = 16584185
        title=event.get('Titre')
        # Scrap data from event page
        headers = { "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:97.0) Gecko/20100101 Firefox/97.0", 
        }
        html_doc = requests.get(url= event_url,headers=headers, timeout=getTimeout).content
        parsed_html = BeautifulSoup(html_doc ,'html.parser')
        
        #Get long description from event page
        long_description=scrap_description(parsed_html)
        if not long_description:
                logging.error(f"Error parsing long_description for event {title}")
                return None
        #Get short description from event page
        description=long_description[:100] if long_description else title
        
        # Get date from CSV
        date= event.get('Date')
        if not date:
                logging.error(f"Error getting date from csv for event {title}")
                return None
        if " au " in date:
                year =date.split(" ")[-1]
                date_begin=date.replace("Du ","").split(" au ")[0].strip() + " " + year
                date_end=date.replace("Du ","").split(" au ")[1].strip()
        else:
                date_begin=date.strip()
                date_end=None
        
        # Get start hour from event page
        start_hour=scrap_start_hour(parsed_html)
                
        # Date Begin: Set complete date & time
        dt_date_begin=scraping_utils.get_datetime_from_text(str(date_begin) + " " + str(start_hour))
        if not dt_date_begin or dt_date_begin.hour is None or dt_date_begin.hour == 0:
                logging.error(f"Error setting hour fo event {title} : {date}")
                return None
        
        # Date END: Set complete date & time
        duree = "2h"  # Default duration
        if not date_end:  # One Day event
                dt_date_end=utils.get_end_date(dt_date_begin, duree)
        else:  # Long event with end date defined
                dt_date_end=scraping_utils.get_datetime_from_text(str(date_end) + " " + str(start_hour))
        logging.info(f"Date End: {dt_date_end.isoformat()}")
        
        ##  Create event dict for OA
        eventOA= {
                        "uid-externe": "scrap-" + event.get('ID') + "-" + slugify(title),
                        "title": { "fr": title  } ,
                        "description": { "fr": description},
                        "locationUid": int(location_uid),
                        "links": event_url,
                        "longDescription": long_description,   
                        "keywords": {
                                "fr": keywords 
                                },
                        "timings": [
                                {
                                "begin": dt_date_begin.isoformat(),
                                "end": dt_date_end.isoformat()
                                },
                                ],
                        "attendanceMode": 3,
                        "onlineAccessLink": event_url,
                }
        return eventOA


In [ ]:
# First create a Json file with all valid events
OAEvents=[]
for event in all_events:
    oa_event = create_event(event)
    if oa_event:
        OAEvents.append(oa_event)
    utils.save_dict_to_json_file(OAEvents, "events2ToPost.json")

In [ ]:
# Then post them and saved them (with attributed unique ID form OA) in a json file
# to update them or restart from last success in case of failing
exit()
saved_events_capv2={}
with open('eventsCapDanse2ToPost.json') as json_file:
    eventsv2 = json.load(json_file)
for event in eventsv2:
    try :
        response = HttpRequests.create_event(access_token,event = event)
        uid = response['event']['uid'] if response['event'] else event['uid-externe'] 
        saved_events_capv2[response['event']['uid'] ] = event
        utils.save_dict_to_json_file(saved_events_capv2, "eventsCapDanse2Created.json")
    except Exception as e:
        print(f"Error creating event {event.get('title')} : {e}")
        continue
    

In [ ]:
# CORRECTIONS SI NECESSAIRE.
# DEFAULT = False
# Update the events with online access link and attendance mode
correction = False
for event in [allMatchEvents[0]]:
    if not correction:
        print("Corrections not executed")
        break
    # print(event.get("title"))
    try:
        response= HttpRequests.search_events(oa.public_key, event.get("title"))
        if not response.get("events"):
            raise ValueError(f"No events return for {event.get("title")} found in OA {response}")
        event= response.get("events")[0]
        if not event.get("uid"):
            raise ValueError(f"Event returned for {event.get("title")} is empty {response.get("events")}")
    except Exception as e:
        print(f"Error in event search: {e}")
        continue
    # pprint.pprint(event)
    eventUID= event.get("uid")
    eventTitle= event.get("title")
    print(eventUID, eventTitle)
    
    #EXEMPLE DE CORRECTION
    # event["onlineAccessLink"] = event.get("lien")
    # event["attendanceMode"] = 3  # 1=offline, 2=online, 3=hybrid
    # event["keywords"] = { "fr" : event.get("keywords")}
    
    HttpRequests.delete_event(
        access_token,
        eventUID
    )
    print("--------")